# 📊 Planilha Embryoscope Combined Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database and **AWS Athena (Production)** to execute data quality and validation queries on the `planilha_embryoscope_combined` table.

It reconciles:
1. **Waterfall Join Match Rates** (Local vs Cloud)
2. **Waterfall Steps Breakdown** (validating 9 priorities of join conditions)
3. **Yearly Match Rates Breakdown** (Planilha-Redlara perspective)
4. **Clinical outcome flags** (`is_transferred_combined`, `has_biopsy`, `has_valid_outcome`, `has_transfer_logging_gap`)
5. **Biopsy Bug Deep Dive** (explaining the 10.71% local vs 100% cloud has_biopsy discrepancy and showing corrected SQL comparison)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Connections setup
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print("Connecting to AWS Athena...")
try:
    ath_con = connect(
        region_name="sa-east-1",
        work_group="datalake-admins"
    )
    ath_cur = ath_con.cursor()
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_cur = None

## 🔍 Part 1: DuckDB Queries (Local Planilha Table)

Running local metrics on `gold.planilha_embryoscope_combined` in DuckDB.

In [ ]:
# 1. DuckDB Overall & Waterfall steps
duck_overall_q = """
SELECT 
    COUNT(*) as total_rows,
    COUNT(join_step) as matched_rows,
    ROUND(COUNT(join_step) * 100.0 / COUNT(*), 2) as match_rate
FROM gold.planilha_embryoscope_combined
"""
df_duck_overall = duck_con.execute(duck_overall_q).df()
print("--- DuckDB Overall Waterfall Stats ---")
display(df_duck_overall)

duck_steps_q = """
SELECT join_step, COUNT(*) as count
FROM gold.planilha_embryoscope_combined
WHERE join_step IS NOT NULL
GROUP BY join_step
ORDER BY join_step
"""
df_duck_steps = duck_con.execute(duck_steps_q).df()
print("\n--- DuckDB Waterfall Steps ---")
display(df_duck_steps)

# 2. DuckDB Yearly Breakdown (Planilha perspective)
duck_yearly_q = """
WITH src_numbered AS (
    SELECT 
        *, 
        ROW_NUMBER() OVER() as src_id,
        COALESCE(fresh_file_name, fet_file_name, 'Unknown') as file_source,
        EXTRACT(YEAR FROM transfer_date) as date_year
    FROM gold.redlara_planilha_combined
),
matched_ids AS (
    SELECT DISTINCT matched_src_row_id FROM gold.planilha_embryoscope_combined WHERE matched_src_row_id IS NOT NULL
),
src_with_year AS (
    SELECT 
        *,
        CASE 
            WHEN file_source LIKE '%2021%' THEN 2021
            WHEN file_source LIKE '%2022%' THEN 2022
            WHEN file_source LIKE '%2023%' THEN 2023
            WHEN file_source LIKE '%2024%' THEN 2024
            WHEN file_source LIKE '%2025%' THEN 2025
            ELSE date_year
        END as year_group
    FROM src_numbered
),
src_stats AS (
    SELECT 
        year_group,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN src_id IN (SELECT matched_src_row_id FROM matched_ids) THEN 1 END) as matched_count
    FROM src_with_year
    GROUP BY 1
)
SELECT 
    CAST(year_group AS INTEGER) as year,
    total_rows,
    matched_count,
    ROUND(CAST(matched_count AS DOUBLE) / total_rows * 100, 2) as match_rate
FROM src_stats
ORDER BY year
"""
df_duck_yearly = duck_con.execute(duck_yearly_q).df()
print("\n--- DuckDB Yearly Breakdown ---")
display(df_duck_yearly)

# 3. DuckDB Clinical Flags
duck_flags_q = """
SELECT 
    COUNT(*) as total_rows,
    COUNT(CASE WHEN is_transferred_combined = True THEN 1 END) as transferred_count,
    ROUND(COUNT(CASE WHEN is_transferred_combined = True THEN 1 END) * 100.0 / COUNT(*), 2) as transferred_pct,
    COUNT(CASE WHEN has_biopsy = True THEN 1 END) as biopsy_count,
    ROUND(COUNT(CASE WHEN has_biopsy = True THEN 1 END) * 100.0 / COUNT(*), 2) as biopsy_pct,
    COUNT(CASE WHEN has_valid_outcome = True THEN 1 END) as valid_outcome_count,
    ROUND(COUNT(CASE WHEN has_valid_outcome = True THEN 1 END) * 100.0 / COUNT(*), 2) as valid_outcome_pct,
    COUNT(CASE WHEN has_transfer_logging_gap = True THEN 1 END) as gap_count,
    ROUND(COUNT(CASE WHEN has_transfer_logging_gap = True THEN 1 END) * 100.0 / COUNT(*), 2) as gap_pct
FROM gold.planilha_embryoscope_combined
"""
df_duck_flags = duck_con.execute(duck_flags_q).df()
print("\n--- DuckDB Clinical Flags ---")
display(df_duck_flags)

## ☁️ Part 2: AWS Athena Queries (Production Planilha Table)

Running metrics on `gold_huntington_prod.planilha_embryoscope_combined` in AWS Athena.

In [ ]:
if ath_cur is not None:
    # 1. Athena Overall & Steps
    ath_overall_q = """
    SELECT 
        COUNT(*) as total_rows,
        COUNT(join_step) as matched_rows,
        ROUND(COUNT(join_step) * 100.0 / COUNT(*), 2) as match_rate
    FROM gold_huntington_prod.planilha_embryoscope_combined
    """
    ath_cur.execute(ath_overall_q)
    df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("--- Athena Overall Waterfall Stats ---")
    display(df_ath_overall)

    ath_steps_q = """
    SELECT join_step, COUNT(*) as count
    FROM gold_huntington_prod.planilha_embryoscope_combined
    WHERE join_step IS NOT NULL
    GROUP BY join_step
    ORDER BY join_step
    """
    ath_cur.execute(ath_steps_q)
    df_ath_steps = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("\n--- Athena Waterfall Steps ---")
    display(df_ath_steps)

    # 2. Athena Yearly Breakdown (Planilha perspective)
    ath_yearly_q = """
    WITH src_numbered AS (
        SELECT 
            *, 
            ROW_NUMBER() OVER() as src_id,
            COALESCE(
                EXTRACT(YEAR FROM transfer_date), 
                EXTRACT(YEAR FROM date_of_embryo_transfer), 
                EXTRACT(YEAR FROM fresh_data_puncao)
            ) as year_group
        FROM gold_huntington_prod.redlara_planilha_combined
    ),
    matched_ids AS (
        SELECT DISTINCT matched_src_row_id FROM gold_huntington_prod.planilha_embryoscope_combined WHERE matched_src_row_id IS NOT NULL
    ),
    src_stats AS (
        SELECT 
            year_group,
            COUNT(*) as total_rows,
            COUNT(CASE WHEN src_id IN (SELECT matched_src_row_id FROM matched_ids) THEN 1 END) as matched_count
        FROM src_numbered
        GROUP BY 1
    )
    SELECT 
        CAST(year_group AS INTEGER) as year,
        total_rows,
        matched_count,
        ROUND(CAST(matched_count AS DOUBLE) / total_rows * 100, 2) as match_rate
    FROM src_stats
    ORDER BY year
    """
    ath_cur.execute(ath_yearly_q)
    df_ath_yearly = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    df_ath_yearly['year'] = df_ath_yearly['year'].astype(float)
    print("\n--- Athena Yearly Breakdown ---")
    display(df_ath_yearly)

    # 3. Athena Clinical Flags
    ath_flags_q = """
    SELECT 
        COUNT(*) as total_rows,
        COUNT(CASE WHEN is_transferred_combined = True THEN 1 END) as transferred_count,
        ROUND(COUNT(CASE WHEN is_transferred_combined = True THEN 1 END) * 100.0 / COUNT(*), 2) as transferred_pct,
        COUNT(CASE WHEN has_biopsy = True THEN 1 END) as biopsy_count,
        ROUND(COUNT(CASE WHEN has_biopsy = True THEN 1 END) * 100.0 / COUNT(*), 2) as biopsy_pct,
        COUNT(CASE WHEN has_valid_outcome = True THEN 1 END) as valid_outcome_count,
        ROUND(COUNT(CASE WHEN has_valid_outcome = True THEN 1 END) * 100.0 / COUNT(*), 2) as valid_outcome_pct,
        COUNT(CASE WHEN has_transfer_logging_gap = True THEN 1 END) as gap_count,
        ROUND(COUNT(CASE WHEN has_transfer_logging_gap = True THEN 1 END) * 100.0 / COUNT(*), 2) as gap_pct
    FROM gold_huntington_prod.planilha_embryoscope_combined
    """
    ath_cur.execute(ath_flags_q)
    df_ath_flags = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("\n--- Athena Clinical Flags ---")
    display(df_ath_flags)
else:
    print("Athena cursor is None. Skipping queries.")

## ⚖️ Part 3: Side-by-Side Reconciliation Tables

Merging and displaying local and cloud statistics side-by-side.

In [ ]:
if ath_cur is not None:
    # 1. Overall comparison
    df_comp_overall = pd.merge(
        df_duck_overall.rename(columns={
            'total_rows': 'duck_total_rows',
            'matched_rows': 'duck_matched',
            'match_rate': 'duck_match_rate'
        }),
        df_ath_overall.rename(columns={
            'total_rows': 'ath_total_rows',
            'matched_rows': 'ath_matched',
            'match_rate': 'ath_match_rate'
        }),
        left_index=True,
        right_index=True
    )
    print("--- Side-by-side Overall Comparison ---")
    display(df_comp_overall)

    # 2. Steps comparison
    df_duck_steps['join_step'] = df_duck_steps['join_step'].astype(float)
    df_ath_steps['join_step'] = df_ath_steps['join_step'].astype(float)
    
    df_comp_steps = pd.merge(
        df_duck_steps.rename(columns={'count': 'duck_count'}),
        df_ath_steps.rename(columns={'count': 'ath_count'}),
        on='join_step',
        how='outer'
    ).sort_values('join_step')
    
    step_labels = {
        1.0: "Punção (Date DL)",
        2.0: "Transfer (Descong)",
        3.0: "Transfer (Trat1)",
        4.0: "Transfer (Trat2)",
        5.0: "Cryo (Redlara)",
        6.0: "Cryo (FET)",
        7.0: "Cryo (Fresh)",
        8.0: "Punção (+/- 3 days)",
        9.0: "Transfer (+/- 3 days)"
    }
    df_comp_steps['step_label'] = df_comp_steps['join_step'].map(step_labels)
    df_comp_steps = df_comp_steps[['join_step', 'step_label', 'duck_count', 'ath_count']]
    print("\n--- Side-by-side Waterfall Steps Comparison ---")
    display(df_comp_steps)

    # 3. Yearly comparison
    df_duck_yearly['year'] = df_duck_yearly['year'].astype(float)
    df_ath_yearly['year'] = df_ath_yearly['year'].astype(float)
    
    df_comp_yearly = pd.merge(
        df_duck_yearly.rename(columns={
            'total_rows': 'duck_total_planilha',
            'matched_count': 'duck_matched',
            'match_rate': 'duck_match_rate'
        }),
        df_ath_yearly.rename(columns={
            'total_rows': 'ath_total_planilha',
            'matched_count': 'ath_matched',
            'match_rate': 'ath_match_rate'
        }),
        on='year',
        how='outer'
    ).sort_values('year')
    print("\n--- Side-by-side Yearly Planilha Match Rates Comparison ---")
    display(df_comp_yearly)

    # 4. Clinical Flags comparison
    df_comp_flags = pd.concat([
        df_duck_flags.rename(index={0: 'Local DuckDB'}),
        df_ath_flags.rename(index={0: 'Cloud Athena'})
    ]).T
    print("\n--- Side-by-side Clinical Flags Comparison ---")
    display(df_comp_flags)

## 🔍 Part 4: Biopsy Bug Deep Dive & Corrected Validation

Evaluating the `has_biopsy` flag discrepancy (10.71% locally vs 100% in Athena).
We show the distribution of values in the source columns to explain the bug, and demonstrate a corrected comparison.

In [ ]:
if ath_cur is not None:
    print("=== ATHENA SOURCE VALUES FOR BIOPSY FIELDS ===")
    ath_cur.execute("""
        SELECT 
            oocito_resultado_pgd IS NULL as pgd_is_null,
            oocito_resultado_pgd = '' as pgd_is_empty,
            oocito_resultado_pgd_detalhes IS NULL as det_is_null,
            oocito_resultado_pgd_detalhes = '' as det_is_empty,
            embryo_description IS NULL as desc_is_null,
            embryo_description = '' as desc_is_empty,
            COUNT(*) as count
        FROM gold_huntington_prod.planilha_embryoscope_combined
        GROUP BY 1, 2, 3, 4, 5, 6
    """)
    df_ath_dist = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    display(df_ath_dist)
    
    # 2. Run corrected biopsy comparison in Athena
    print("\n=== CORRECTED BIOPSY COUNT IN ATHENA ===")
    print("Running corrected SQL logic: excluding empty strings '' in addition to NULL and 'None'")
    
    corrected_biopsy_q = """
    SELECT 
        COUNT(*) as total_rows,
        COUNT(CASE WHEN 
            (oocito_resultado_pgd IS NOT NULL AND oocito_resultado_pgd != 'None' AND oocito_resultado_pgd != '') OR
            (oocito_resultado_pgd_detalhes IS NOT NULL AND oocito_resultado_pgd_detalhes != 'None' AND oocito_resultado_pgd_detalhes != '') OR
            (embryo_description IS NOT NULL AND embryo_description != 'None' AND embryo_description != '')
        THEN 1 END) as corrected_biopsy_count
    FROM gold_huntington_prod.planilha_embryoscope_combined
    """
    ath_cur.execute(corrected_biopsy_q)
    df_corrected = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    df_corrected['corrected_biopsy_pct'] = (df_corrected['corrected_biopsy_count'] * 100.0 / df_corrected['total_rows']).round(2)
    display(df_corrected)
    
    # Print the local biopsy stats for comparison
    print("\nLocal DuckDB biopsy stats for reference:")
    display(df_duck_flags[['total_rows', 'biopsy_count', 'biopsy_pct']])